# PyDI Data Integration Workflow: Videogames

This notebook demonstrates comprehensive data integration using PyDI. We'll work with vidoegame datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 0: Schema Matching and Normalization](#part-0-schema-matching-and-normalization)
- [Part 1: Data Profiling](#part-1-data-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 65,000 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

## Part 0: Schema Matching and Normalization

In [44]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR  # Base directory for all input files
DATA_DIR = NOTEBOOK_DIR / "data"
SCHEMA_DIR = NOTEBOOK_DIR / "schemamatching"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "games"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [45]:
import pandas as pd
import json
from PyDI.schemamatching import SchemaTranslator
from PyDI.normalization import load_normalization_spec

## Step 1: Load Target Schema and Normalization Spec

In [46]:
# Load the JSON Schema (used for both matching and normalization)
with open(SCHEMA_DIR / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema (properties at root level)
spec = load_normalization_spec(SCHEMA_DIR / "target_schema.json")

# Clear taxonomy settings (we don't want to use taxonomy normalization here)
for col_name, col_spec in spec.columns.items():
    col_spec.taxonomy_path = None
    col_spec.taxonomy_column = None
    col_spec.taxonomy_mapping_path = None

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,releaseYear,datetime
3,developer,string
4,publisher,string
5,platform,string
6,criticScore,float
7,userScore,float
8,ESRB,string
9,globalSales,int


## Step 2: Load Source Datasets

In [47]:
dbpedia = pd.read_csv(DATA_DIR / "regulat_headers_csv" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia.head()

,wiki_ref,name,releaseYear,developer,platform,series
0,dbpedia_1,San Francisco Rush 2049,2006-01-01,Handheld Games,Game Boy Color,Rush (video game series)
1,dbpedia_2,RoboCop (1988 video game),1989-01-01,Ocean Software,Arcade video game,List of RoboCop video games
2,dbpedia_3,Air (video game),2016-01-01,Key (company),PlayStation Vita,NaN
3,dbpedia_4,Fallout 2,1998-01-01,Black Isle Studios,Mac OS X,Fallout (series)
4,dbpedia_5,SpongeBob SquarePants: Creature from the Krust...,2006-01-01,Blitz Games,Wii,SpongeBob SquarePants video games


In [48]:
# Drop duplicates in noisy dbpedia dataset
# drop duplicates that share name, platform, developer and releaseYear
dbpedia = dbpedia.drop_duplicates(subset=["name", "platform", "developer", "releaseYear"])
len(dbpedia)

46580

In [49]:
metacritic = pd.read_csv(DATA_DIR / "regulat_headers_csv" / "metacritic.csv")
metacritic.attrs["dataset_name"] = "metacritic"
metacritic.head()

,mc_id,name,releaseYear,developer,platform,criticScore,userScore,ESRB
0,metacritic_1,Red Dead Redemption 2,2018-01-01,Rockstar Games,Xbox One,97.0,8.3,M
1,metacritic_2,Grand Theft Auto IV,2008-01-01,Rockstar North,Xbox 360,98.0,8.0,M
2,metacritic_3,SoulCalibur,1999-01-01,Namco,Dreamcast,98.0,8.4,T
3,metacritic_4,Tony Hawk's Pro Skater 2,2000-01-01,Neversoft Entertainment,PlayStation,98.0,7.5,T
4,metacritic_5,Super Mario Galaxy,2007-01-01,Nintendo,Wii,97.0,9.1,E


In [50]:
sales = pd.read_csv(DATA_DIR / "regulat_headers_csv" / "sales.csv")
sales.attrs["dataset_name"] = "sales"
sales.head()

,Attribute_1,name,releaseYear,developer,publisher,platform,criticScore,userScore,ESRB,sales
0,sales_1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,76,8.0,E,82
1,sales_2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,82,8.3,E,35
2,sales_3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,80,8.0,E,32
3,sales_4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,89,8.5,E,29
4,sales_5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,58,6.6,E,28


In [51]:
# Create id columns based on index (starting with 1)
sales["id"] = sales.index + 1
metacritic["id"] = metacritic.index + 1
dbpedia["id"] = dbpedia.index + 1

## Step 3: Manual Schema Mapping

Define mappings from each source dataset to the target schema columns.

In [52]:
# Manual mapping for DBpedia
# Source columns: wiki_ref, name, releaseYear, developer, platform, series
dbpedia_mapping = pd.DataFrame([
    {"source_dataset": "dbpedia", "source_column": "name", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "Game title"},
    {"source_dataset": "dbpedia", "source_column": "platform", "target_dataset": "target_schema", "target_column": "platform", "score": 1.0, "notes": "Gaming platform"},
    {"source_dataset": "dbpedia", "source_column": "developer", "target_dataset": "target_schema", "target_column": "developer", "score": 1.0, "notes": "Developer company"},
    {"source_dataset": "dbpedia", "source_column": "releaseYear", "target_dataset": "target_schema", "target_column": "releaseYear", "score": 1.0, "notes": "Release year"},
    {"source_dataset": "dbpedia", "source_column": "series", "target_dataset": "target_schema", "target_column": "series", "score": 1.0, "notes": "Game franchise/series"},
])
dbpedia_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,name,target_schema,name,1.0,Game title
1,dbpedia,platform,target_schema,platform,1.0,Gaming platform
2,dbpedia,developer,target_schema,developer,1.0,Developer company
3,dbpedia,releaseYear,target_schema,releaseYear,1.0,Release year
4,dbpedia,series,target_schema,series,1.0,Game franchise/series


In [53]:
# Manual mapping for Metacritic
# Source columns: mc_id, name, releaseYear, developer, platform, criticScore, userScore, ESRB
metacritic_mapping = pd.DataFrame([
    {"source_dataset": "metacritic", "source_column": "name", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "Game title"},
    {"source_dataset": "metacritic", "source_column": "releaseYear", "target_dataset": "target_schema", "target_column": "releaseYear", "score": 1.0, "notes": "Release year"},
    {"source_dataset": "metacritic", "source_column": "developer", "target_dataset": "target_schema", "target_column": "developer", "score": 1.0, "notes": "Developer company"},
    {"source_dataset": "metacritic", "source_column": "platform", "target_dataset": "target_schema", "target_column": "platform", "score": 1.0, "notes": "Gaming platform"},
    {"source_dataset": "metacritic", "source_column": "ESRB", "target_dataset": "target_schema", "target_column": "ESRB", "score": 1.0, "notes": "ESRB rating code"},
    {"source_dataset": "metacritic", "source_column": "criticScore", "target_dataset": "target_schema", "target_column": "criticScore", "score": 1.0, "notes": "Critic score (0-100)"},
    {"source_dataset": "metacritic", "source_column": "userScore", "target_dataset": "target_schema", "target_column": "userScore", "score": 1.0, "notes": "User score (0-10)"},
])
metacritic_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,metacritic,name,target_schema,name,1.0,Game title
1,metacritic,releaseYear,target_schema,releaseYear,1.0,Release year
2,metacritic,developer,target_schema,developer,1.0,Developer company
3,metacritic,platform,target_schema,platform,1.0,Gaming platform
4,metacritic,ESRB,target_schema,ESRB,1.0,ESRB rating code
5,metacritic,criticScore,target_schema,criticScore,1.0,Critic score (0-100)
6,metacritic,userScore,target_schema,userScore,1.0,User score (0-10)


In [54]:
# Manual mapping for Sales
# Source columns: Attribute_1, name, releaseYear, developer, publisher, platform, criticScore, userScore, ESRB, sales
sales_mapping = pd.DataFrame([
    {"source_dataset": "sales", "source_column": "name", "target_dataset": "target_schema", "target_column": "name", "score": 1.0, "notes": "Game title"},
    {"source_dataset": "sales", "source_column": "platform", "target_dataset": "target_schema", "target_column": "platform", "score": 1.0, "notes": "Gaming platform"},
    {"source_dataset": "sales", "source_column": "releaseYear", "target_dataset": "target_schema", "target_column": "releaseYear", "score": 1.0, "notes": "Release year"},
    {"source_dataset": "sales", "source_column": "publisher", "target_dataset": "target_schema", "target_column": "publisher", "score": 1.0, "notes": "Publisher company"},
    {"source_dataset": "sales", "source_column": "sales", "target_dataset": "target_schema", "target_column": "globalSales", "score": 1.0, "notes": "Global sales"},
    {"source_dataset": "sales", "source_column": "criticScore", "target_dataset": "target_schema", "target_column": "criticScore", "score": 1.0, "notes": "Critic score (0-100)"},
    {"source_dataset": "sales", "source_column": "userScore", "target_dataset": "target_schema", "target_column": "userScore", "score": 1.0, "notes": "User score (0-10)"},
    {"source_dataset": "sales", "source_column": "developer", "target_dataset": "target_schema", "target_column": "developer", "score": 1.0, "notes": "Developer company"},
    {"source_dataset": "sales", "source_column": "ESRB", "target_dataset": "target_schema", "target_column": "ESRB", "score": 1.0, "notes": "ESRB rating code"},
])
sales_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,sales,name,target_schema,name,1.0,Game title
1,sales,platform,target_schema,platform,1.0,Gaming platform
2,sales,releaseYear,target_schema,releaseYear,1.0,Release year
3,sales,publisher,target_schema,publisher,1.0,Publisher company
4,sales,sales,target_schema,globalSales,1.0,Global sales
5,sales,criticScore,target_schema,criticScore,1.0,Critic score (0-100)
6,sales,userScore,target_schema,userScore,1.0,User score (0-10)
7,sales,developer,target_schema,developer,1.0,Developer company
8,sales,ESRB,target_schema,ESRB,1.0,ESRB rating code


## Step 4: Translate and Normalize


In [55]:
# Unify platform names across datasets
platform_groups = {
    "Nintendo Entertainment System": ["NES"],
    "Super Nintendo": ["SNES", "Super Nintendo Entertainment System"],
    "Nintendo 64": ["N64"],
    "GameCube": ["Nintendo GameCube", "GC"],
    "Wii": ["Nintendo Wii"],
    "Game Boy Color": ["GBC"],
    "Game Boy Advance": ["GBA"],
    "Game Boy": ["GB"],
    "DS": ["Nintendo DS"],
    "3DS": ["Nintendo 3DS"],
    "Switch": ["Nintendo Switch"],

    "Playstation": [
        "Playstation (console)", "PlayStation (console)",
        "Playstation 1", "PlayStation 1",
        "PS1", "PSX", "PS"
    ],

    "PS2": ["Playstation 2", "PlayStation 2"],
    "PS3": ["Playstation 3", "PlayStation 3"],
    "PS4": ["Playstation 4", "PlayStation 4"],
    "Playstation Portable": ["PSP"],
    "Playstation Vita": ["PSV", "PS Vita"],
    "Playstation VR": ["PSVR", "PS VR"],

    "Xbox": ["XB", "Xbox (console)"],
    "Xbox One": ["XOne"],
    "Xbox 360": ["X360"],

    "PC": ["Microsoft Windows", "Windows"],
}

platform_map = {}
for canonical, variants in platform_groups.items():
    for alias in variants:
        platform_map[alias.lower()] = canonical

def normalize_platform(series):
    def _norm(x):
        # Leave missing or non-string values as they are
        if not isinstance(x, str):
            return x
        key = x.strip().lower()
        return platform_map.get(key, x.strip())
    
    return series.apply(_norm)

dbpedia["platform"] = normalize_platform(dbpedia["platform"])
metacritic["platform"] = normalize_platform(metacritic["platform"])
sales["platform"] = normalize_platform(sales["platform"])

In [56]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("releaseYear", output_type="datetime")
dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse special date format (e.g., Oct 26, 2018)
spec.set_column("releaseYear", output_type="datetime", date_format="%b %d, %Y")

metacritic_normalized = translator.translate(
    metacritic, metacritic_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse year-only values (e.g., "1929", "2010")
spec.set_column("releaseYear", output_type="datetime", date_format="%Y")

sales_normalized = translator.translate(
    sales, sales_mapping,
    normalize=spec, on_failure="keep"
)

[INFO ] root - Translating 5 columns for 'dbpedia'
[WARNING] PyDI.normalization.transform - Column 'publisher' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'criticScore' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'userScore' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'ESRB' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'globalSales' not found in DataFrame
[INFO ] root - Normalization complete: 45171 values transformed, 0 values failed
[INFO ] root - Translating 7 columns for 'metacritic'
[WARNING] PyDI.normalization.transform - Column 'publisher' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'globalSales' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'series' not found in DataFrame
[INFO ] root - Normalization complete: 0 values transformed, 20494 values failed
[INFO ] root - Translating 9 columns for 'sales'
[WARNING] PyDI.normali

In [57]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head(10)

,id,name,releaseYear,developer,platform,series
0,1,San Francisco Rush 2049,2006-01-01,Handheld Games,Game Boy Color,Rush (video game series)
1,2,RoboCop (1988 video game),1989-01-01,Ocean Software,Arcade video game,List of RoboCop video games
2,3,Air (video game),2016-01-01,Key (company),PlayStation Vita,NaN
3,4,Fallout 2,1998-01-01,Black Isle Studios,Mac OS X,Fallout (series)
4,5,SpongeBob SquarePants: Creature from the Krust...,2006-01-01,Blitz Games,Wii,SpongeBob SquarePants video games
5,6,Transformers: Fall of Cybertron,2016-01-01,High Moon Studios,PS4,Transformers
6,7,List of Monster Jam video games,2003-01-01,Ubi Soft Barcelona,Xbox One,Monster Jam
7,8,Onimusha: Warlords,2002-01-01,Capcom,Xbox One,Onimusha
8,9,Nicktoons: Battle for Volcano Island,2006-01-01,Halfbrick,PS2,SpongeBob SquarePants video games
9,10,Grid Autosport,2019-01-01,Feral Interactive,Xbox 360,Grid (series)


In [58]:
metacritic_cols = [c for c in target_columns if c in metacritic_normalized.columns]
metacritic_normalized[metacritic_cols].head(10)

,id,name,releaseYear,developer,platform,criticScore,userScore,ESRB
0,1,Red Dead Redemption 2,2018-01-01,Rockstar Games,Xbox One,97.0,8.3,M
1,2,Grand Theft Auto IV,2008-01-01,Rockstar North,Xbox 360,98.0,8.0,M
2,3,SoulCalibur,1999-01-01,Namco,Dreamcast,98.0,8.4,T
3,4,Tony Hawk's Pro Skater 2,2000-01-01,Neversoft Entertainment,PlayStation,98.0,7.5,T
4,5,Super Mario Galaxy,2007-01-01,Nintendo,Wii,97.0,9.1,E
5,6,Grand Theft Auto IV,2008-01-01,Rockstar North,PS3,98.0,7.9,M
6,7,Call of Duty 4: Modern Warfare,2007-01-01,Infinity Ward,Xbox 360,94.0,8.5,M
7,8,The Elder Scrolls IV: Oblivion,2006-01-01,"Bethesda Softworks,Bethesda Game Studios",PC,94.0,8.3,M
8,9,Super Mario Galaxy 2,2010-01-01,Nintendo EAD Tokyo,Wii,97.0,9.1,E
9,10,The Legend of Zelda: Ocarina of Time,1998-01-01,Nintendo,Nintendo 64,99.0,9.0,E


In [59]:
sales_cols = [c for c in target_columns if c in sales_normalized.columns]
sales_normalized[sales_cols].head(10)

,id,name,releaseYear,developer,publisher,platform,criticScore,userScore,ESRB,globalSales
0,1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,76.0,8.0,E,82
1,2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,82.0,8.3,E,35
2,3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,80.0,8.0,E,32
3,4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,89.0,8.5,E,29
4,5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,58.0,6.6,E,28
5,6,New Super Mario Bros. Wii,2009-01-01,Nintendo,Nintendo,Wii,87.0,8.4,E,28
6,7,Mario Kart DS,2005-01-01,Nintendo,Nintendo,DS,91.0,8.6,E,23
7,8,Wii Fit,2007-01-01,Nintendo,Nintendo,Wii,80.0,7.7,E,22
8,9,Kinect Adventures!,2010-01-01,Good Science Studio,Microsoft Game Studios,Xbox 360,61.0,6.3,E,21
9,10,Wii Fit Plus,2009-01-01,Nintendo,Nintendo,Wii,80.0,7.4,E,21


In [60]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
metacritic = metacritic_normalized[metacritic_cols].copy()
sales = sales_normalized[sales_cols].copy()

# Create proper id entries for datasets
dbpedia["id"] = dbpedia["id"].apply(lambda x: f"dbpedia_{x}")
metacritic["id"] = metacritic["id"].apply(lambda x: f"metacritic_{x}")
sales["id"] = sales["id"].apply(lambda x: f"sales_{x}")

## Part 1: Data Profiling

In [61]:
# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 74,951


In [62]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

dbpedia:
  Rows: 46,580
  Columns: 6
  Total nulls: 25,233
  Null percentage: 9.0%
  Null counts per column:
    releaseYear: 1,409 (3.0%)
    developer: 1,221 (2.6%)
    platform: 410 (0.9%)
    series: 22,193 (47.6%)

metacritic:
  Rows: 20,494
  Columns: 8
  Total nulls: 3,718
  Null percentage: 2.3%
  Null counts per column:
    developer: 19 (0.1%)
    criticScore: 10 (0.0%)
    userScore: 1,413 (6.9%)
    ESRB: 2,276 (11.1%)

sales:
  Rows: 7,877
  Columns: 10
  Total nulls: 1,053
  Null percentage: 1.3%
  Null counts per column:
    publisher: 1 (0.0%)
    userScore: 1,052 (13.4%)



### Attribute Coverage Analysis

In [63]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 11 attributes across 3 datasets


📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,metacritic_count,metacritic_pct,metacritic_coverage,metacritic_samples,sales_count,sales_pct,sales_coverage,sales_samples,avg_coverage,max_coverage,datasets_with_attribute
0,ESRB,0/0,0%,0.000000,N/A,18218/20494,88.9%,0.888943,"['M', 'M', 'T']",7877/7877,100.0%,1.000000,"['E', 'E', 'E']",0.629648,1.000000,2
1,criticScore,0/0,0%,0.000000,N/A,20484/20494,100.0%,0.999512,"[97.0, 98.0, 98.0]",7877/7877,100.0%,1.000000,"[76.0, 82.0, 80.0]",0.666504,1.000000,2
2,developer,45359/46580,97.4%,0.973787,"['Handheld Games', 'Ocean Software', 'Key (com...",20475/20494,99.9%,0.999073,"['Rockstar Games', 'Rockstar North', 'Namco']",7877/7877,100.0%,1.000000,"['Nintendo', 'Nintendo', 'Nintendo']",0.990953,1.000000,3
3,globalSales,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7877/7877,100.0%,1.000000,"[82, 35, 32]",0.333333,1.000000,1
4,id,46580/46580,100.0%,1.000000,"['dbpedia_1', 'dbpedia_2', 'dbpedia_3']",20494/20494,100.0%,1.000000,"['metacritic_1', 'metacritic_2', 'metacritic_3']",7877/7877,100.0%,1.000000,"['sales_1', 'sales_2', 'sales_3']",1.000000,1.000000,3
5,name,46580/46580,100.0%,1.000000,"['San Francisco Rush 2049', 'RoboCop (1988 vid...",20494/20494,100.0%,1.000000,"['Red Dead Redemption 2', 'Grand Theft Auto IV...",7877/7877,100.0%,1.000000,"['Wii Sports', 'Mario Kart Wii', 'Wii Sports R...",1.000000,1.000000,3
6,platform,46170/46580,99.1%,0.991198,"['Game Boy Color', 'Arcade video game', 'PlayS...",20494/20494,100.0%,1.000000,"['Xbox One', 'Xbox 360', 'Dreamcast']",7877/7877,100.0%,1.000000,"['Wii', 'Wii', 'Wii']",0.997066,1.000000,3
7,publisher,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7876/7877,100.0%,0.999873,"['Nintendo', 'Nintendo', 'Nintendo']",0.333291,0.999873,1
8,releaseYear,45171/46580,97.0%,0.969751,"[Timestamp('2006-01-01 00:00:00'), Timestamp('...",20494/20494,100.0%,1.000000,"['2018-01-01', '2008-01-01', '1999-01-01']",7877/7877,100.0%,1.000000,"['2006-01-01', '2008-01-01', '2009-01-01']",0.989917,1.000000,3
9,series,24387/46580,52.4%,0.523551,"['Rush (video game series)', 'List of RoboCop ...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.174517,0.523551,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']


## Part 2: Entity Matching

### Step 1: Blocking

In [64]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [65]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

dbpedia['name_longest_token'] = dbpedia['name'].apply(get_longest_token)
metacritic['name_longest_token'] = metacritic['name'].apply(get_longest_token)
sales['name_longest_token'] = sales['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    metacritic, dbpedia,
    on=['name_longest_token', 'platform'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_m2d = standard_blocker_m2d.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 13482 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 18788 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5915 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv


### Step 2: Evaluate Blocking Against Ground Truth

In [66]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 0 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 0 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 0 true matches


[INFO ] root - Processed 40 batches, 40000 pairs, 0 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 0 true matches
[INFO ] root -   Pair Completeness: 0.000
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   0.999940
[INFO ] root -   True Matches Found: 0/106
[INFO ] root -   Batches Processed:  57
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.0,
 'pair_quality': 0.0,
 'reduction_ratio': 0.9999403924440304,
 'total_candidates': 56902,
 'total_possible_pairs': 954610520,
 'true_positives_found': 0,
 'total_true_pairs': 106,
 'batches_processed': 57,
 'evaluation_timestamp': '2026-02-04T20:10:01.881590',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/blocking_detailed_results.csv']}

In [67]:
standard_blocker_m2s = StandardBlocker(
    metacritic, sales,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2s,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5497 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2231 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2100 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv


[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 0 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 0 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 0 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 0 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 0 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 0 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 0 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 0 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 0 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 0 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 0 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 0 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 0 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 0 true matches
[INFO ] root - Processed 

{'pair_completeness': 0.0,
 'pair_quality': 0.0,
 'reduction_ratio': 0.9989540500209755,
 'total_candidates': 168849,
 'total_possible_pairs': 161431238,
 'true_positives_found': 0,
 'total_true_pairs': 116,
 'batches_processed': 169,
 'evaluation_timestamp': '2026-02-04T20:10:02.652202',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/blocking-evaluation/blocking_detailed_results.csv']}

Now let's evaluate which blocking method we want to use for each dataset combination:

### Step 3: Entity Matching with Comparators

In [68]:
from PyDI.entitymatching import StringComparator, DateComparator

# Create comparators for different attributes
comparators_m2d = [
    # Name similarity - most important for games
    StringComparator(
        column='name',
        similarity_function='jaccard',  # Good for game names
        preprocess=str.lower  # Case normalization
    ),
    
    # Platform similarity - supporting evidence
    StringComparator(
        column='developer',
        similarity_function='jaccard',
        preprocess=str.lower
    ),

    # Date proximity - games from same year likely same game
    DateComparator(
        column='releaseYear'
    )
]

comparators_m2s = [
    StringComparator(
        column='name',
        similarity_function='jaccard',
        preprocess=str.lower
    ),
        # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
    ),
    DateComparator(
        column='releaseYear',
        max_days_difference=360  # Allow almost 1 year difference
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [69]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=metacritic,
    df_right=dbpedia, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators_m2d,
    weights=[0.6, 0.3, 0.1], # name, developer, releaseYear
    threshold=0.9, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 46580 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 46580 elements after 0:00:0.018; 56902 blocked pairs (reduction ratio: 0.9999403924440304)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:10.228; found 3482 correspondences.


In [70]:
correspondences_m2s = matcher.match(
    df_left=metacritic,
    df_right=sales, 
    candidates=standard_blocker_m2s,
    comparators=comparators_m2s,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 7877 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 7877 elements after 0:00:0.031; 168849 blocked pairs (reduction ratio: 0.9989540500209755)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:44.538; found 6575 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [71]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  0
[INFO ] root -   True Negatives:  231
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 106
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.685
[INFO ] root -   Precision: 0.000
[INFO ] root -   Recall:    0.000
[INFO ] root -   F1-Score:  0.000


{'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'accuracy': 0.685459940652819,
 'true_positives': 0,
 'false_positives': 0,
 'false_negatives': 106,
 'true_negatives': 231,
 'threshold_used': 0.0,
 'total_correspondences': 3482,
 'filtered_correspondences': 3482,
 'evaluation_timestamp': '2026-02-04T20:10:59.428285',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/debug_results_entity_matching/matching_detailed_results.csv']}

In [72]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2742 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2232	|	81.40%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	342	|	12.47%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	122	|	4.45%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	36	|	1.31%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	8	|	0.29%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	2	|	0.07%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/cluster_analysis/cluster_size_distribution.csv


Analyzing cluster size distribution in our entity matching results...


In [73]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2742 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [74]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
     
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 3482 -> 3482 (threshold=0.0)
[INFO ] root - Greedy matching: 3482 -> 2744 correspondences (5488 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 3482 -> 2744 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 6224 -> 5488 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2744 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2744	|	100.00%
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  0
[INFO ] root -   True Negatives:  231
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 106
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.685
[INFO ] root -   Precision: 0.000
[INFO ] root -   Recall:    0.000
[INFO ] root -   F1-Score:  0.000


In [75]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2s = clusterer.cluster(correspondences_m2s)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  0
[INFO ] root -   True Negatives:  286
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 116
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.711
[INFO ] root -   Precision: 0.000
[INFO ] root -   Recall:    0.000
[INFO ] root -   F1-Score:  0.000
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 6346 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	6249	|	98.47%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	35	|	0.55%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	55	|	0.87%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	2	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	5	|	0.08%
[INFO ] root - Filtered correspondences: 6575 -> 6575 (threshold=0.0)
[INFO ] root - Maximum bipar

## Part 3: Data Fusion

In [76]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 9,157


## Step 1: Define Fusion Strategy 

In [77]:
from PyDI.fusion import DataFusionStrategy, longest_string, prefer_higher_trust, voting, average

strategy = DataFusionStrategy('game_fusion_strategy')
['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', longest_string)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('userScore', average)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'platform' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'developer' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'releaseYear' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'ESRB' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'criticScore' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'userScore' using rule 'average'


## Step 2: Run Fusion

In [78]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=True,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'game_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 16942 of 16942 unique IDs
[INFO ] PyDI.fusion.engine - Created 65794 record groups from 9157 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 65794 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	6413	|	9.75%
[INFO ] PyDI.fusion.engine - 		3	|	1372	|	2.09%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     ESRB: 1.00
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.en

Fused rows: 65,794


,_id,_fusion_sources,_fusion_source_datasets,platform,name,userScore,criticScore,ESRB,releaseYear,globalSales,metacritic_id,name_longest_token,developer,publisher,id,_fusion_confidence,_fusion_metadata,series
0,sales_1152,"[sales_1152, metacritic_2398]","[sales, metacritic]",Xbox,The Lord of the Rings: The Return of the King,8.60,84.0,T,2003-01-01,1.0,metacritic_2398,Return,EA Games,Electronic Arts,sales_1152,0.767442,"{'platform_rule': 'voting', 'platform_sources'...",NaN
1,dbpedia_18439,"[dbpedia_18439, metacritic_2428]","[dbpedia, metacritic]",PC,SpeedRunners,7.60,84.0,None,2017-01-01 00:00:00,NaN,metacritic_2428,SpeedRunners,DoubleDutch Games,NaN,dbpedia_18439,0.600000,"{'platform_rule': 'voting', 'platform_sources'...",None
2,dbpedia_10282,"[dbpedia_10282, metacritic_9948]","[dbpedia, metacritic]",PC,Hard Reset,7.30,73.0,M,2011-01-01 00:00:00,NaN,metacritic_9948,Reset,Flying Wild Hog,NaN,dbpedia_10282,0.700000,"{'platform_rule': 'voting', 'platform_sources'...",None
3,metacritic_10653,"[metacritic_10653, sales_6138]","[metacritic, sales]",PC,Gray Matter,7.65,72.0,T,2011-01-01,0.0,metacritic_10653,Matter,Wizarbox,DTP Entertainment,metacritic_10653,0.772133,"{'platform_rule': 'voting', 'platform_sources'...",NaN
4,metacritic_11141,"[metacritic_11141, sales_7325]","[metacritic, sales]",PS2,Graffiti Kingdom,7.10,71.0,E,2005-01-01,0.0,metacritic_11141,Graffiti,Taito Corporation,505 Games,metacritic_11141,0.727273,"{'platform_rule': 'voting', 'platform_sources'...",NaN


## Step 3: Evaluate Data Fusion

In [79]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match, numeric_tolerance_match, exact_match
strategy.add_evaluation_function("name", exact_match)
strategy.add_evaluation_function("platform", exact_match)
strategy.add_evaluation_function("developer", exact_match)
strategy.add_evaluation_function("releaseYear", year_only_match)
strategy.add_evaluation_function("ESRB", exact_match)
strategy.add_evaluation_function("criticScore", numeric_tolerance_match, tolerance=2)
strategy.add_evaluation_function("userScore", numeric_tolerance_match, tolerance=0.2)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'platform'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'developer'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'releaseYear'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'ESRB'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'criticScore' with params {'tolerance': 2}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'userScore' with params {'tolerance': 0.2}


In [80]:
# Evaluate fusion results against validation set in order to make adjustments if necessary
from PyDI.fusion import DataFusionEvaluator
from PyDI.io import load_xml

fusion_val_set = load_xml(INPUT_DIR / 'fusion' / 'validation_set.xml', name='fusion_val_set', nested_handling='aggregate')
# transform releaseYear column to datetime
fusion_val_set['releaseYear'] = pd.to_datetime(fusion_val_set['releaseYear'],errors='coerce')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the validation set
print("Evaluating fusion results against validation set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_val_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Validation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation


Evaluating fusion results against validation set...


[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.912 overall accuracy (73/80)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 7 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |       3 |     42.86%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |       2 |     28.57%%
[INFO ] PyDI.fusion.evaluation - 	ESRB                             |       1 |     14.29%%
[INFO ] PyDI.fusion.evaluation - 	developer                        |       1 |     14.29%%



Fusion Validation Results:
  overall_accuracy: 0.912
  macro_accuracy: 0.913
  num_evaluated_records: 10
  num_evaluated_attributes: 8
  total_evaluations: 80
  total_correct: 73
  platform_accuracy: 1.000
  platform_count: 10
  userScore_accuracy: 0.700
  userScore_count: 10
  name_accuracy: 1.000
  name_count: 10
  criticScore_accuracy: 1.000
  criticScore_count: 10
  ESRB_accuracy: 0.900
  ESRB_count: 10
  releaseYear_accuracy: 1.000
  releaseYear_count: 10
  developer_accuracy: 0.900
  developer_count: 10
  publisher_accuracy: 0.800
  publisher_count: 10

Overall Accuracy: 91.2%


In [81]:
# Finally, evaluate against test set
fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
fusion_test_set['releaseYear'] = pd.to_datetime(fusion_test_set['releaseYear'],errors='coerce')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the test set
print("Evaluating fusion results against test set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Test Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation


Evaluating fusion results against test set...


[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.840 overall accuracy (100/119)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 19 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |      10 |     52.63%%
[INFO ] PyDI.fusion.evaluation - 	developer                        |       3 |     15.79%%
[INFO ] PyDI.fusion.evaluation - 	name                             |       2 |     10.53%%
[INFO ] PyDI.fusion.evaluation - 	ESRB                             |       2 |     10.53%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |       2 |     10.53%%



Fusion Test Results:
  overall_accuracy: 0.840
  macro_accuracy: 0.836
  num_evaluated_records: 15
  num_evaluated_attributes: 8
  total_evaluations: 119
  total_correct: 100
  platform_accuracy: 1.000
  platform_count: 15
  userScore_accuracy: 0.286
  userScore_count: 14
  name_accuracy: 0.867
  name_count: 15
  criticScore_accuracy: 1.000
  criticScore_count: 15
  ESRB_accuracy: 0.867
  ESRB_count: 15
  releaseYear_accuracy: 1.000
  releaseYear_count: 15
  developer_accuracy: 0.800
  developer_count: 15
  publisher_accuracy: 0.867
  publisher_count: 15

Overall Accuracy: 84.0%


## Part 5: Reports and Metrics

In [82]:
# Source Overview Report
from PyDI.pipeline.reporting import save_source_overview

# Save source overview (using normalized datasets before ID prefixing)
sources = {
    "dbpedia": dbpedia,
    "metacritic": metacritic,
    "sales": sales,
}

REPORTS_DIR = OUTPUT_DIR / "reporting"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

source_overview_path = save_source_overview(
    sources=sources,
    output_dir=REPORTS_DIR,
    target_columns=len(target_columns),
)
print(f"Source overview saved to: {source_overview_path}")
pd.read_csv(source_overview_path)

Source overview saved to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/reporting/source_overview.csv


,Source,Rows,Columns,Data Density,Density in Target Schema (11 cols)
0,Dbpedia,46580,7,92.26%,58.71% (7/11 × 92.26%)
1,Metacritic,20494,10,98.19%,89.26% (10/11 × 98.19%)
2,Sales,7877,11,98.78%,98.78% (11/11 × 98.78%)
3,Total Input,74951,11 (target),96.41% avg,71.28% weighted avg


In [83]:
# Schema Matching Report
from PyDI.pipeline.reporting import save_schema_matching_report, generate_column_mapping_table

SCHEMA_MATCHING_DIR = REPORTS_DIR / "schema_matching"
SCHEMA_MATCHING_DIR.mkdir(parents=True, exist_ok=True)

# Prepare mappings dictionary
mappings = {
    "dbpedia": dbpedia_mapping,
    "metacritic": metacritic_mapping,
    "sales": sales_mapping,
}

# Generate column mapping table
mapping_table = generate_column_mapping_table(mappings, target_schema)
mapping_table.to_csv(SCHEMA_MATCHING_DIR / "column_mapping.csv", index=False)
print(f"Schema matching report saved to: {SCHEMA_MATCHING_DIR}")
mapping_table

Schema matching report saved to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/reporting/schema_matching


,Target Column,Dbpedia,Metacritic,Sales
0,id,-,-,-
1,name,name,name,name
2,releaseYear,releaseYear,releaseYear,releaseYear
3,developer,developer,developer,developer
4,genres,-,-,-
5,publisher,-,-,publisher
6,platform,platform,platform,platform
7,criticScore,-,criticScore,criticScore
8,userScore,-,userScore,userScore
9,ESRB,-,ESRB,ESRB


In [84]:
# Entity Matching Summary
MATCHING_DIR = REPORTS_DIR / "entity_matching"
MATCHING_DIR.mkdir(parents=True, exist_ok=True)

# Evaluate both entity matching pairs and save summaries
pairs_data = []

# Metacritic to DBpedia
gt_test_m2d = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)
eval_m2d = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test_m2d,
    out_dir=MATCHING_DIR
)
pairs_data.append({
    "left": "metacritic",
    "right": "dbpedia", 
    "best_matcher": "RuleBasedMatcher",
    "f1": eval_m2d["f1"],
    "precision": eval_m2d["precision"],
    "recall": eval_m2d["recall"],
    "threshold": 0.9,
})

# Metacritic to Sales
gt_test_m2s = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)
eval_m2s = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test_m2s,
    out_dir=MATCHING_DIR
)
pairs_data.append({
    "left": "metacritic",
    "right": "sales",
    "best_matcher": "RuleBasedMatcher",
    "f1": eval_m2s["f1"],
    "precision": eval_m2s["precision"],
    "recall": eval_m2s["recall"],
    "threshold": 0.8,
})

# Save matching summary
matching_summary = pd.DataFrame(pairs_data)
matching_summary.to_csv(MATCHING_DIR / "matching_summary.csv", index=False)
print(f"Entity matching summary saved to: {MATCHING_DIR / 'matching_summary.csv'}")
matching_summary

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  0
[INFO ] root -   True Negatives:  231
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 106
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.685
[INFO ] root -   Precision: 0.000
[INFO ] root -   Recall:    0.000
[INFO ] root -   F1-Score:  0.000
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  0
[INFO ] root -   True Negatives:  286
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 116
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.711
[INFO ] root -   Precision: 0.000
[INFO ] root -   Recall:    0.000
[INFO ] root -   F1-Score:  0.000


Entity matching summary saved to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/reporting/entity_matching/matching_summary.csv


,left,right,best_matcher,f1,precision,recall,threshold
0,metacritic,dbpedia,RuleBasedMatcher,0.0,0.0,0.0,0.9
1,metacritic,sales,RuleBasedMatcher,0.0,0.0,0.0,0.8


In [85]:
# Fusion Evaluation Summary
FUSION_DIR = REPORTS_DIR / "fusion"
FUSION_DIR.mkdir(parents=True, exist_ok=True)

# Save fusion comparison results
fusion_summary_data = []

# Add manual/heuristic results (validation and test)
fusion_summary_data.append({
    "variant": "manual",
    "case": "heuristic",
    "validation_accuracy": None,  # Validation was done earlier but not saved here
    "test_accuracy": evaluation_results["overall_accuracy"],
    "test_correct": evaluation_results["total_correct"],
    "test_total": evaluation_results["total_evaluations"],
})

# Save to CSV
fusion_summary = pd.DataFrame(fusion_summary_data)
fusion_summary.to_csv(FUSION_DIR / "fusion_comparison_summary.csv", index=False)
print(f"Fusion comparison saved to: {FUSION_DIR / 'fusion_comparison_summary.csv'}")

# Display per-attribute accuracy
per_attr_cols = [k for k in evaluation_results.keys() if k.endswith("_accuracy")]
per_attr_data = {k.replace("_accuracy", ""): [evaluation_results[k]] for k in per_attr_cols}
pd.DataFrame(per_attr_data)

Fusion comparison saved to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/reporting/fusion/fusion_comparison_summary.csv


,overall,macro,platform,userScore,name,criticScore,ESRB,releaseYear,developer,publisher
0,0.840336,0.835714,1.0,0.285714,0.866667,1.0,0.866667,1.0,0.8,0.866667


In [86]:
# End-to-End Metrics
from PyDI.pipeline.end_to_end_metrics import EndToEndMetrics, SourceStats, calculate_density, save_end_to_end_report

# Calculate per-source statistics
per_source_stats = []
for name, df in sources.items():
    density = calculate_density(df)
    cols = [c for c in df.columns if not c.startswith("_")]
    per_source_stats.append(SourceStats(
        name=name,
        rows=len(df),
        columns=len(cols),
        density=density,
    ))

# Calculate merged records count (records from 2+ sources)
merged_count = 0
if "_fusion_source_datasets" in fused.columns:
    for val in fused["_fusion_source_datasets"]:
        if isinstance(val, list) and len(val) > 1:
            merged_count += 1
        elif isinstance(val, str) and val not in ("", "[]"):
            try:
                import json as json_module
                parsed = json_module.loads(val)
                if isinstance(parsed, list) and len(parsed) > 1:
                    merged_count += 1
            except:
                if len([s.strip() for s in val.split(",") if s.strip()]) > 1:
                    merged_count += 1

# Calculate metrics
total_input_rows = sum(s.rows for s in per_source_stats)
max_source = max(per_source_stats, key=lambda s: s.rows)
fused_density = calculate_density(fused)
fused_cols = [c for c in fused.columns if not c.startswith("_fusion_")]

# Get all unique columns across sources
all_columns = set()
for df in sources.values():
    all_columns.update(c for c in df.columns if not c.startswith("_"))

# Calculate average input density
total_non_null = sum(df[[c for c in df.columns if not c.startswith("_")]].notna().sum().sum() for df in sources.values())
total_cells = sum(len(df) * len(all_columns) for df in sources.values())
avg_input_density = total_non_null / total_cells if total_cells > 0 else 0.0

e2e_metrics = EndToEndMetrics(
    num_sources=len(sources),
    total_input_rows=total_input_rows,
    total_input_columns=len(all_columns),
    avg_input_density=avg_input_density,
    per_source_stats=per_source_stats,
    fused_rows=len(fused),
    fused_columns=len(fused_cols),
    fused_density=fused_density,
    max_source_rows=max_source.rows,
    row_gain_over_largest=len(fused) - max_source.rows,
    row_gain_pct=((len(fused) - max_source.rows) / max_source.rows * 100) if max_source.rows > 0 else 0.0,
    largest_source_name=max_source.name,
    largest_source_density=max_source.density,
    merged_records=merged_count,
    density_change=fused_density - avg_input_density,
)

# Save reports
txt_path, csv_path = save_end_to_end_report(e2e_metrics, REPORTS_DIR)
print(f"End-to-end report saved to: {csv_path}")

# Also save JSON format
import json
json_path = REPORTS_DIR / "end_to_end_metrics.json"
json_path.write_text(json.dumps(e2e_metrics.to_dict(), indent=2))

# Display the summary table
e2e_metrics.summary_table()

End-to-end report saved to: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/input/games/output/games/reporting/end_to_end_report.csv


,Metric,Value
0,Input Sources,3
1,Total Input Records,"74,951"
2,Fused Output Records,"65,794"
3,Fusion Ratio,87.8%
4,Row Gain Over Largest Source,"+19,214"
5,Row Gain Percentage,+41.2%
6,Merged Records,"7,785"
7,Input Columns (Target Schema),13
8,Fused Output Columns,14
9,Average Input Data Density,60.3%
